# 3. Running the tests on your own ensemble

A template: set the paths below to your own tables and run from the top. It is set up for the CMIP6
example so that it runs as it stands. The file format is described in `docs/new_ensemble.md`; in short:

* a values CSV with a `model` column and one column per quantity (name the β/γ convention, e.g.
  `gamma_land_coubgc`);
* a flags CSV with a `model` column, one 0/1 column per process (e.g. `demography`) and, optionally, a
  `family` column.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from tcre_evidence import process_plots as pp, process_tests as pt
from tcre_evidence.paths import DATA

pd.set_option("display.precision", 2)

NAME = "cmip6"                    # a name for this ensemble's cache, e.g. "wiemip"
VALUES = DATA / "examples" / "cmip6_land_feedbacks.csv"
FLAGS = DATA / "examples" / "cmip6_process_flags.csv"
QUANTITY = "gamma_land_coubgc"    # a column of VALUES
KIND = "additive"                 # "additive" for gamma (PgC/K), "multiplicative" for beta (PgC/ppm)
LABEL = "γ_L (PgC/K)"

## Look at the data

In [ ]:
y, flags, family = pt.load_ensemble(VALUES, FLAGS, QUANTITY)
pp.by_flag(y, flags, family, LABEL);

## Quick tests (seconds)

In [ ]:
yy = np.log(y) if KIND == "multiplicative" else y   # multiplicative effects are tested on log values
b = pt.flag_budget(yy, flags)
print(f"flags explain {b['r2']:.0%} of the spread; predicting each model from the others: {b['loo_q2']:.2f}")
effects = {p: 0.5 * yy.std() for p in flags.columns}        # effect sizes to test; edit as you like
pp.power(pt.detection_power(yy, flags, effects));

## Bayesian tests

These need sampling. Run the precompute script in a terminal (the next cell prints the command); it takes
from a few minutes (`--quick`) to about an hour. Then run the cells below it.

In [ ]:
print(f"python scripts/precompute.py --name {NAME} --values {VALUES} --flags {FLAGS} "
      f"--quantities {QUANTITY}:{KIND} --jobs 4")

In [ ]:
res = pp.load(NAME, QUANTITY)
pp.prior_posterior(res);

In [ ]:
pp.recovery(res);
pt.evidence_table(res["prior_vs_posterior"], res["synthetic_effects_summary"],
                  res["synthetic_no_effects_summary"], res["power"])

In [ ]:
pp.loo(res, LABEL);
res["loo_summary"]

## Checks before quoting a number

* `res["run"]["diagnostics"]` should show `rhat_max` below 1.01 and few or no divergences.
* Results from `--quick` runs are for exploring only.
* `res["run"]` records the priors, effect sizes, settings and package versions used.

In [ ]:
res["run"]["diagnostics"]